In [ ]:
import sys
sys.path.append("agentic_rag")

In [ ]:
import wikipedia as wk
wk.set_lang("en")
wk.wikipedia.USER_AGENT = "Project (YOUR-MAIL-ID)"

In [20]:
search = "Linear Algebra"
result = wk.search(search, results = 2)
print(search)
print(result)

Linear Algebra
['Linear algebra', 'Trace (linear algebra)']


In [ ]:
summary = wk.summary(search)
print(summary[:500])

In [22]:
type(summary)

str

In [ ]:
full_page = wk.page(search)
full_page.content

In [40]:
print(full_page.title)
print(full_page.pageid)
print(full_page.categories)
print(full_page.original_title)

Linear algebra
18422
['All pages needing cleanup', 'Articles needing cleanup from September 2018', 'Articles with short description', 'Commons category link from Wikidata', 'Linear algebra', 'Numerical analysis', 'Pages that use a deprecated format of the math tags', 'Short description is different from Wikidata', 'Wikipedia pages needing cleanup from September 2018']
Linear algebra


# Pull documents

In [5]:
from src.data_pipeline.pull_wikipedia import pull_corpus

corpus = pull_corpus(output_path="articles")

Saved 49 articles


In [ ]:
print(len(corpus))          
print(corpus[0])           
lengths = [len(c['content']) for c in corpus]
print(f"Min: {min(lengths)}, Max: {max(lengths)}, Avg: {sum(lengths)/len(lengths):.0f}")

# Langchain Document

In [ ]:
from src.data_pipeline.chunking import langchain_docs

document = langchain_docs(corpus_path="agentic_rag/data/raw/articles.json")

In [4]:
document[44].metadata

{'title': 'Distance', 'pageid': '39378'}

In [5]:
document[44].metadata['title']

'Distance'

# Chunking

In [7]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

encoder = "o200k_base"
encoding = tiktoken.get_encoding(encoding_name=encoder)

In [8]:
tokens = encoding.encode(document[44].metadata['title'])
print(f"{document[44].metadata['title']}----> {tokens}")

Distance----> [17525]


In [9]:
example_doc = document[44].page_content[:120]
text_split = RecursiveCharacterTextSplitter(chunk_size = 6,
                                            chunk_overlap = 2,
                                            length_function = lambda text: len(encoding.encode(text)))
chunks = text_split.split_text(example_doc)
chunks

['Distance is a numerical or occasionally',
 'or occasionally qualitative measurement of how',
 'of how far apart objects,',
 'objects, points, people,',
 'people, or ideas are.',
 'are.']

In [10]:
from src.data_pipeline.chunking import CHUNK

chk = CHUNK()
encoding = chk.load_encoder(encoder="o200k_base")
chunks = chk.text_splitter(chunk_size=6, chunk_overlap=2, encoding=encoding)
chunks.split_text(example_doc)

['Distance is a numerical or occasionally',
 'or occasionally qualitative measurement of how',
 'of how far apart objects,',
 'objects, points, people,',
 'people, or ideas are.',
 'are.']

## chunking documents

In [6]:
from src.data_pipeline.chunking import CHUNK

chk = CHUNK()
encoding = chk.load_encoder(encoder="o200k_base")
chunks = chk.text_splitter(chunk_size=500, chunk_overlap=50, encoding=encoding)
chunk = chunks.split_documents(document)
chunk[0]

Document(metadata={'title': 'Linear algebra', 'pageid': '18422'}, page_content='Linear algebra is the branch of mathematics concerning linear equations such as\n\n  \n    \n      \n        \n          a\n          \n            1\n          \n        \n        \n          x\n          \n            1\n          \n        \n        +\n        ⋯\n        +\n        \n          a\n          \n            n\n          \n        \n        \n          x\n          \n            n\n          \n        \n        =\n        b\n        ,\n      \n    \n    {\\displaystyle a_{1}x_{1}+\\cdots +a_{n}x_{n}=b,}\n  \n\nlinear maps such as\n\n  \n    \n      \n        (\n        \n          x\n          \n            1\n          \n        \n        ,\n        …\n        ,\n        \n          x\n          \n            n\n          \n        \n        )\n        ↦\n        \n          a\n          \n            1\n          \n        \n        \n          x\n          \n            1\n          \n    

In [7]:
print(len(chunk))

2180


In [ ]:
import json

with open("agentic_rag/data/chunks/chunks.json", "w") as f:
    json.dump([{"page_content": doc.page_content,
                "metadata": doc.metadata} for doc in chunk],
              f,indent=2 )

# Embedding

In [9]:
import warnings
warnings.filterwarnings('ignore')

from src.models.embeddings import load_model

embedding = load_model(model_name='all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14421.60it/s]


Embedding Dimension : 384


# Vectorstore and Retrieval

In [ ]:
from src.models.vectorstore import build_vectorstore

vs = build_vectorstore(chunks=chunk, embedding=embedding, directory="agentic_rag/data/chroma_db")

collection count : 2180


In [12]:
from src.tools.retriever_tool import make_retriever_tool

retrive = make_retriever_tool(vectorstore=vs, k=3)
retrive

StructuredTool(name='search_math_knowledge', description='Search a knowledge base of math theorems, definitions, and concepts.\n\n        Use this tool when the user asks about a definition, theorem statement,\n        or conceptual explanation of a mathematical topic (e.g., "what is the\n        Cauchy-Schwarz inequality", "explain eigenvalues").\n\n        Args:\n            query: The user\'s question or topic to search for.\n\n        Returns:\n            Relevant text passages from the knowledge base.', args_schema=<class 'langchain_core.utils.pydantic.search_math_knowledge'>, func=<function make_retriever_tool.<locals>.search_math_knowledge at 0x7447eea9d900>)

# SYMPY

In [3]:
import sympy

x = sympy.symbols('x')
sympy.Eq((x+1)**2 - x**2 - 1, 0)

Eq(-x**2 + (x + 1)**2 - 1, 0)

In [2]:
from src.tools.compute_tool import equations

print(equations("x**2 + 2*x + 1 - (x+1)**2", mode="simplify"))  
print(equations("x**2 - 4", mode="solve"))  

0
[-2, 2]


# Langchain Agent - Example

In [2]:
from langchain.agents import create_agent
from src.agent.agent import CLIENT

llm = CLIENT()

def check_weather(location: str) -> str:
    '''Return the weather forecast for the specified location.'''
    return f"It's always sunny in {location}"

graph = create_agent(
    model=llm.get_llm(),
    tools=[check_weather],
    system_prompt="You are a helpful assistant",
)
inputs = {"messages": [{"role": "user", "content": "what is the weather in New Jersey"}]}

In [3]:
for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 138, 'total_tokens': 197, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-aa78d6bddf402fa5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a02387-d846-7ef1-888a-129fdbb2ecf0-0', tool_calls=[{'name': 'check_weather', 'args': {'location': 'New Jersey'}, 'id': 'chatcmpl-tool-aca5346f78e7cec2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 59, 'total_tokens': 197, 'input_token_details': {}, 'output_token_details': {}})]}}
{'tools': {'messages': [ToolMessage(content="It's always sunny in New Jersey", name='check_weather', id='afceecad-105c-4ca1-b0ea-680e68606c58', tool_call_id='chatcmpl-tool-aca5346f78e7cec2')]}}
{'model': {'messages': [AIMessa

In [6]:
for chunk in graph.stream(inputs, stream_mode="updates"):

    for node, value in chunk.items():

        if node == "model":
            message = value["messages"][-1]

            if message.tool_calls:
                for tool_call in message.tool_calls:
                    print("Tool:", tool_call["name"])

            elif message.content:
                print("Final answer:", message.content)

        elif node == "tools":
            message = value["messages"][-1]
            print("Tool result:", message.content)

Tool: check_weather
Tool result: It's always sunny in New Jersey
Final answer: The latest forecast for New Jersey is: **It’s always sunny in New Jersey**.


In [18]:
for value in chunk.values():
    print(value)

{'messages': [AIMessage(content='Tamil\u202fNadu generally enjoys a warm, tropical climate.\u202fDuring this time of year the state is typically experiencing the tail‑end of the summer season, so you can expect:\n\n* **Temperature:** Daytime highs usually range from about\u202f30\u202f°C to\u202f35\u202f°C (86‑95\u202f°F), with nighttime lows in the mid‑20\u202f°C (mid‑60\u202f°F).  \n* **Sky conditions:** The weather is often clear to partly cloudy, and the recent forecast indicates mostly sunny skies across the region.  \n* **Humidity:** Relative humidity is moderate, often around\u202f50‑70\u202f%, which can make the heat feel a bit more intense.  \n* **Rainfall:** The monsoon season hasn’t fully arrived yet, so rain is generally scarce, though occasional showers can pop up, especially in the western hills.\n\nIf you need a more precise update for a particular city (e.g., Chennai, Coimbatore, Madurai) or a specific time, let me know and I can pull the latest observations for that lo

In [20]:
if "messages" in value:
    message = value['messages'][-1]
    if message.type == 'ai':
        print(message.content)

Tamil Nadu generally enjoys a warm, tropical climate. During this time of year the state is typically experiencing the tail‑end of the summer season, so you can expect:

* **Temperature:** Daytime highs usually range from about 30 °C to 35 °C (86‑95 °F), with nighttime lows in the mid‑20 °C (mid‑60 °F).  
* **Sky conditions:** The weather is often clear to partly cloudy, and the recent forecast indicates mostly sunny skies across the region.  
* **Humidity:** Relative humidity is moderate, often around 50‑70 %, which can make the heat feel a bit more intense.  
* **Rainfall:** The monsoon season hasn’t fully arrived yet, so rain is generally scarce, though occasional showers can pop up, especially in the western hills.

If you need a more precise update for a particular city (e.g., Chennai, Coimbatore, Madurai) or a specific time, let me know and I can pull the latest observations for that location.


# Agent

In [23]:
from src.agent.agent import AGENT, CLIENT
from src.tools.compute_tool import equations
from src.tools.retriever_tool import make_retriever_tool

client = CLIENT()
llm = client.get_llm()
search_math_knowledge = make_retriever_tool(vs, k=3) 

agent = AGENT(llm=llm, tools=[equations, search_math_knowledge])

agent.ask("What is vector space? give definition and contitions alone.")

**Vector Space (over a field \(F\))**

A **vector space** is a set \(V\) together with two operations  

1. **Addition**: \(+\colon V\times V\to V\)  
2. **Scalar multiplication**: \(\cdot\colon F\times V\to V\)

that satisfy the following eight axioms for all vectors \(\mathbf{u},\mathbf{v},\mathbf{w}\in V\) and all scalars \(\alpha,\beta\in F\):

1. **Associativity of addition**  
   \((\mathbf{u}+\mathbf{v})+\mathbf{w}= \mathbf{u}+(\mathbf{v}+\mathbf{w})\).

2. **Commutativity of addition**  
   \(\mathbf{u}+\mathbf{v}= \mathbf{v}+\mathbf{u}\).

3. **Existence of additive identity**  
   There exists a vector \(\mathbf{0}\in V\) such that \(\mathbf{v}+\mathbf{0}= \mathbf{v}\) for every \(\mathbf{v}\in V\).

4. **Existence of additive inverses**  
   For each \(\mathbf{v}\in V\) there exists a vector \(-\mathbf{v}\in V\) with \(\mathbf{v}+(-\mathbf{v})= \mathbf{0}\).

5. **Compatibility of scalar multiplication with field multiplication**  
   \((\alpha\beta)\mathbf{v}= \alpha(\beta\

In [24]:
agent.ask("Is x**2 + 2*x + 1 equal to (x+1)**2?")


Yes.

\[
(x+1)^2 = (x+1)(x+1)=x\cdot x + x\cdot 1 + 1\cdot x + 1\cdot 1
        = x^{2}+2x+1.
\]

Since the difference \((x^{2}+2x+1) - (x+1)^{2}\) simplifies to \(0\), the two expressions are identically equal for every value of \(x\).


# Question-Tool pair Generation

In [2]:
topics = ["Linear algebra","Vector space","Matrix multiplication","Eigenvalues and eigenvectors","Singular value decomposition",
    "Principal component analysis","Multivariable calculus","Partial derivative","Gradient","Jacobian matrix",
    "Hessian matrix","Chain rule","Taylor series","Mathematical optimization","Convex optimization","Gradient descent",
    "Probability theory","Conditional probability","Bayes' theorem","Random variable","Probability distribution",
    "Normal distribution","Bernoulli distribution","Binomial distribution","Expected value","Variance",
    "Covariance","Correlation","Law of large numbers","Central limit theorem","Maximum likelihood estimation",
    "Bayesian inference","Statistical estimation","Statistical hypothesis testing","Confidence interval",
    "Regression analysis","Linear regression","Logistic regression","Regularization (mathematics)",
    "Entropy (information theory)","Cross-entropy","Kullback–Leibler divergence","Mutual information",
    "Information gain","Distance","Euclidean distance","Manhattan distance","Norm (mathematics)","Inner product space"
]


In [ ]:
from src.evaluation.build_eval_set import EVALUATION_SET
from src.agent.agent import CLIENT


client = CLIENT()
llm = client.get_llm()

evaluation = EVALUATION_SET()

evaluation_set = evaluation.build_evaluation_set(topics=topics,
                                                 output_path="/agentic_rag/data/eval",
                                                 llm=llm,
                                                 no_of_samples=50)

Generated 50 QA pairs


# Evaluation

In [12]:
from src.agent.agent import AGENT, CLIENT
from src.tools.compute_tool import equations
from src.tools.retriever_tool import make_retriever_tool

client = CLIENT()
llm = client.get_llm()
search_math_knowledge = make_retriever_tool(vs, k=3) 

agent = AGENT(llm=llm, tools=[equations, search_math_knowledge])

result = agent.ask_with_trace("If a fair six-sided die is rolled, what is the probability that the outcome is even given that it is greater than 3?")
print(result)

{'question': 'If a fair six-sided die is rolled, what is the probability that the outcome is even given that it is greater than 3?', 'tools_called': [], 'final_answer': 'The conditional probability  \n\n\\[\nP(\\text{even} \\mid \\text{>\u202f3})=\\frac{P(\\text{even and >\u202f3})}{P(\\text{>\u202f3})}.\n\\]\n\nFor a fair six‑sided die the possible outcomes are \\(\\{1,2,3,4,5,6\\}\\).\n\n* Outcomes that are **greater than 3**: \\(\\{4,5,6\\}\\).  \n  So \\(P(\\text{>\u202f3}) = \\frac{3}{6}= \\frac12\\).\n\n* Outcomes that are **both even and greater than 3**: \\(\\{4,6\\}\\).  \n  So \\(P(\\text{even and >\u202f3}) = \\frac{2}{6}= \\frac13\\).\n\nNow\n\n\\[\nP(\\text{even} \\mid \\text{>\u202f3}) = \\frac{\\frac13}{\\frac12}= \\frac{2}{3}.\n\\]\n\n\\[\n\\boxed{\\displaystyle \\frac{2}{3}}\n\\]'}


In [ ]:
import json

with open("agentic_rag/data/eval/evaluation_set.json", "r") as f:
    eval_set = json.load(f)

results = []
for item in eval_set:
    try:
        trace = agent.ask_with_trace(item["question"])
        results.append({
            "question": item["question"],
            "expected_tool": item["expected_tool"],
            "tools_called": trace["tools_called"],
            "final_answer": trace["final_answer"],
            "error": None
        })
    except Exception as e:
        results.append({
            "question": item["question"],
            "expected_tool": item["expected_tool"],
            "tools_called": [],
            "final_answer": None,
            "error": str(e)
        })

with open("agentic_rag/data/eval/eval_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Ran {len(results)} questions, {sum(1 for r in results if r['error'])} errors")

Ran 50 questions, 4 errors


In [ ]:
from src.evaluation.run_evaluation import *

with open("agentic_rag/data/eval/eval_results.json", "r") as f:
    results = json.load(f)

metrics = score_evaluation(results)
print(metrics)

{'total': 50, 'accuracy': 0.34, 'breakdown': {'no_tool': 27, 'correct': 17, 'error': 4, 'wrong_tool': 2}}


# Result Improvement

In [13]:
from src.agent.agent import AGENT, CLIENT
from src.tools.compute_tool import equations
from src.tools.retriever_tool import make_retriever_tool

client = CLIENT()
llm = client.get_llm()
search_math_knowledge = make_retriever_tool(vs, k=3) 

agent = AGENT(llm=llm, tools=[equations, search_math_knowledge])

In [ ]:
import json

with open("agentic_rag/data/eval/evaluation_set.json", "r") as f:
    eval_set = json.load(f)

results = []
for item in eval_set:
    try:
        trace = agent.ask_with_trace(item["question"])
        results.append({
            "question": item["question"],
            "expected_tool": item["expected_tool"],
            "tools_called": trace["tools_called"],
            "final_answer": trace["final_answer"],
            "error": None
        })
    except Exception as e:
        results.append({
            "question": item["question"],
            "expected_tool": item["expected_tool"],
            "tools_called": [],
            "final_answer": None,
            "error": str(e)
        })

with open("agentic_rag/data/eval/eval_results_v2.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Ran {len(results)} questions, {sum(1 for r in results if r['error'])} errors")

Ran 50 questions, 9 errors


In [ ]:
from src.evaluation.run_evaluation import *

with open("agentic_rag/data/eval/eval_results_v2.json", "r") as f:
    results = json.load(f)

metrics = score_evaluation(results)
print(metrics)

{'total': 50, 'accuracy': 0.62, 'breakdown': {'wrong_tool': 10, 'correct': 31, 'error': 9}}
